In [ ]:
import os
import re
import pandas as pd
import numpy as np

def extract_axis_features_from_folder(folder_path):
    features_list = []
    file_names = []
    action_list = []
    name_list = []

    col_labels = ['Acc_X', 'Acc_Y', 'Acc_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z']

    for file in os.listdir(folder_path):
        if file.endswith(".csv"):
            file_path = os.path.join(folder_path, file)
            try:
                df = pd.read_csv(file_path)
                if df.shape[1] >= 7:
                    data = df.iloc[:, 1:7]
                elif df.shape[1] == 6:
                    data = df.iloc[:, :6]
                else:
                    print(f"Skipping {file}: Not enough columns found ({df.shape[1]} columns).")
                    continue

                data = data.apply(pd.to_numeric, errors='coerce')
                if data.empty:
                    print(f"Skipping {file}: Data frame is empty after selection.")
                    continue

                means = data.mean().values
                stds = data.std().values
                mins = data.min().values
                maxs = data.max().values
                feature_vector = np.concatenate([means, stds, mins, maxs])

                features_list.append(feature_vector)
                file_names.append(file)

                # parse Action and Name from filename (handles underscores, spaces or dashes)
                base = os.path.splitext(file)[0]
                parts = re.split('[ _-]+', base)
                action = parts[0] if len(parts) >= 1 else ''
                name = parts[1] if len(parts) >= 2 else ''
                action_list.append(action)
                name_list.append(name)

            except Exception as e:
                print(f"Error processing {file}: {e}")

    stats = ['Mean', 'Std', 'Min', 'Max']
    final_headers = [f"{stat}_{col}" for stat in stats for col in col_labels]

    result_df = pd.DataFrame(features_list, columns=final_headers)
    result_df.insert(0, 'Action', action_list)
    result_df.insert(1, 'Name', name_list)

    return result_df

# --- USAGE SECTION ---

# 1. CHANGE THIS PATH to the folder containing your Phyphox CSV files
my_folder_path = './HumanActivity'

if os.path.exists(my_folder_path):
    # 2. Run the function
    features_df = extract_axis_features_from_folder(my_folder_path)

    # Print the result (First 5 rows)
    print("\nFeatures extracted successfully:")
    print(features_df.head())
    
    # 3. Optional: Save the features to a new CSV file ready for training
    output_file_name = 'extracted_features.csv'
    features_df.to_csv(output_file_name)
    print(f"\nFeatures saved to {output_file_name}")
else:
    print(f"ERROR: The specified folder path does not exist: {my_folder_path}")


Features extracted successfully:
      Action    Name  Mean_Acc_X  Mean_Acc_Y  Mean_Acc_Z  Mean_Gyro_X  \
0  ChairSpin  Haakon   -0.442745   -1.736867   -9.644374    -0.023778   
1  ChairSpin  Yvette    4.726099   -7.136783   -4.794481     1.111467   
2    Jumping  Haakon   -5.891960   -7.634484   -3.685534     0.010030   
3    Jumping  Yvette   -0.516850   -9.795802   -0.777187     0.005048   
4     Squats  Haakon   -4.941832   -4.978509   -4.623544     0.022613   

   Mean_Gyro_Y  Mean_Gyro_Z  Std_Acc_X  Std_Acc_Y  ...  Min_Acc_Z  Min_Gyro_X  \
0    -0.017344    -0.354468   1.033262   0.948789  ... -14.811131   -2.181247   
1    -1.585301    -1.431357   0.532527   0.635750  ...  -6.602994    0.670730   
2    -0.118189    -0.067262   9.606667  13.088459  ... -78.244904  -18.832823   
3    -0.042934    -0.004224   6.159481  12.710528  ... -20.048265   -3.510334   
4     0.019917    -0.011743   2.507410   3.820190  ... -15.505448   -5.352936   

   Min_Gyro_Y  Min_Gyro_Z  Max_Acc_X  Ma